## Proyecto – Data Analysis with Pandas and Streamlit

*Puedes utilizar este notebook para analisar el dataset propuesto

In [1]:
import tarfile
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import clear_output, display, Markdown


def load_data():
    archive_path = next(
        (folder / "data" / "california_housing.tgz"
         for folder in (Path.cwd(), *Path.cwd().parents)
         if (folder / "data" / "california_housing.tgz").exists()),
        None,
    )
    if archive_path is None:
        raise FileNotFoundError(
            "No se encontró data/california_housing.tgz. "
            "Abrí el notebook desde la carpeta del proyecto."
        )

    with tarfile.open(archive_path, mode="r:gz") as archive:
        data_file = archive.extractfile("CaliforniaHousing/cal_housing.data")
        if data_file is None:
            raise FileNotFoundError("El archivo de datos no está dentro del archivo California Housing.")
        housing = np.loadtxt(data_file, delimiter=",")

    housing = housing[:, [8, 7, 2, 3, 4, 5, 6, 1, 0]]
    target = housing[:, 0] / 100000.0
    features = housing[:, 1:]
    features[:, 2] /= features[:, 5]
    features[:, 3] /= features[:, 5]
    features[:, 5] = features[:, 4] / features[:, 5]
    columns = [
        "MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population",
        "AveOccup", "Latitude", "Longitude",
    ]
    frame = pd.DataFrame(features, columns=columns)
    frame["MedHouseVal"] = target
    return frame


df_california = load_data()
display(Markdown("### California Housing"))
display(Markdown(f"{len(df_california):,} registros; {df_california.shape[1]} columnas."))
display(df_california.head())
display(Markdown("#### Valores faltantes por columna"))
display(df_california.isna().sum().to_frame("Valores faltantes"))


def mostrar_analisis(rango_edad, latitud_minima):
    filtrados = df_california[
        df_california["HouseAge"].between(*rango_edad)
        & (df_california["Latitude"] >= latitud_minima)
    ]
    clear_output(wait=True)
    display(Markdown("### Análisis filtrado"))
    display(Markdown(f"**Registros:** {len(filtrados):,} de {len(df_california):,}"))
    if filtrados.empty:
        display(Markdown("No hay datos para esos filtros."))
        return

    mediana = filtrados["MedHouseVal"].median()
    valor_rango = filtrados["MedHouseVal"].max() - filtrados["MedHouseVal"].min()
    display(Markdown(f"**Mediana del valor:** ${mediana:.2f} (unidades de 100 mil)"))
    display(Markdown(f"**Rango:** ${valor_rango:.2f}"))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(filtrados["MedHouseVal"], bins=30, kde=True, color="skyblue", ax=axes[0])
    axes[0].set(title="Distribución del valor de vivienda", xlabel="MedHouseVal ($100k)", ylabel="Frecuencia")
    sns.scatterplot(data=filtrados, x="MedInc", y="MedHouseVal", alpha=0.35, color="purple", ax=axes[1])
    axes[1].set(title="Ingresos vs. valor de vivienda", xlabel="Mediana de ingresos", ylabel="MedHouseVal")
    plt.tight_layout()
    display(fig)
    plt.close(fig)

    puntos = filtrados.sample(min(3000, len(filtrados)), random_state=42)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(puntos["Longitude"], puntos["Latitude"], s=5, alpha=0.4)
    ax.set(title="Ubicación de los bloques censales", xlabel="Longitud", ylabel="Latitud")
    display(fig)
    plt.close(fig)


controles = widgets.interactive(
    mostrar_analisis,
    rango_edad=widgets.FloatRangeSlider(
        value=(float(df_california["HouseAge"].min()), float(df_california["HouseAge"].max())),
        min=float(df_california["HouseAge"].min()),
        max=float(df_california["HouseAge"].max()),
        step=1.0,
        description="Edad",
        continuous_update=False,
        layout=widgets.Layout(width="600px"),
    ),
    latitud_minima=widgets.FloatSlider(
        value=float(df_california["Latitude"].min()),
        min=float(df_california["Latitude"].min()),
        max=float(df_california["Latitude"].max()),
        step=0.1,
        description="Latitud mín.",
        continuous_update=False,
        layout=widgets.Layout(width="600px"),
    ),
)
display(controles)

### California Housing

20,640 registros; 9 columnas.

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


#### Valores faltantes por columna

,Valores faltantes
MedInc,0
HouseAge,0
AveRooms,0
AveBedrms,0
Population,0
AveOccup,0
Latitude,0
Longitude,0
MedHouseVal,0


interactive(children=(FloatRangeSlider(value=(1.0, 52.0), continuous_update=False, description='Edad', layout=…